# Day 4: Fund Performance Analytics

This notebook loads the cleaned fund data, calculates performance metrics, compares top funds against benchmark indices, and saves outputs for the Bluestock Mutual Fund Analytics Capstone.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

## Paths and data sources

In [ ]:
ROOT = Path('..').resolve()
RAW_DIR = ROOT / 'data' / 'raw'
PROCESSED_DIR = ROOT / 'data' / 'processed'
REPORTS_DIR = ROOT / 'reports'
CHARTS_DIR = REPORTS_DIR / 'charts'
REPORTS_DIR.mkdir(exist_ok=True, parents=True)
CHARTS_DIR.mkdir(exist_ok=True, parents=True)

## Load the data

In [ ]:
def load_csv(name, filename):
    raw_path = RAW_DIR / filename
    processed_path = PROCESSED_DIR / filename
    if processed_path.exists():
        df = pd.read_csv(processed_path)
    elif raw_path.exists():
        df = pd.read_csv(raw_path)
    else:
        raise FileNotFoundError(f'Missing dataset: {filename}')
    df.columns = [str(col).strip().lower().replace(' ', '_') for col in df.columns]
    return df

fund_master = load_csv('fund_master', '01_fund_master.csv')
nav_history = load_csv('nav_history', '02_nav_history.csv')
scheme_performance = load_csv('scheme_performance', '07_scheme_performance.csv')
benchmark_indices = load_csv('benchmark_indices', '10_benchmark_indices.csv')

fund_master.head()

## Compute daily returns and performance metrics

In [ ]:
nav_history['date'] = pd.to_datetime(nav_history['date'], errors='coerce')
nav_history = nav_history.sort_values(['amfi_code', 'date'])
nav_history['daily_return'] = nav_history.groupby('amfi_code')['nav'].pct_change()
nav_history[['amfi_code', 'date', 'nav', 'daily_return']].head()

In [ ]:
def annualized_sharpe(return_series, risk_free=0.065):
    returns = return_series.dropna()
    if returns.empty or returns.std() == 0:
        return np.nan
    mean_ann = returns.mean() * 252
    std_ann = returns.std() * np.sqrt(252)
    return (mean_ann - risk_free) / std_ann

def annualized_sortino(return_series, risk_free=0.065):
    returns = return_series.dropna()
    downside = returns[returns < 0]
    if returns.empty or downside.empty:
        return np.nan
    mean_ann = returns.mean() * 252
    downside_dev = np.sqrt((downside ** 2).mean()) * np.sqrt(252)
    return (mean_ann - risk_free) / downside_dev

metrics = nav_history.groupby('amfi_code')['daily_return'].apply(lambda s: pd.Series({
    'sharpe_ratio': annualized_sharpe(s),
    'sortino_ratio': annualized_sortino(s),
})).reset_index()
metrics.head()

## Recent CAGR snapshots

In [ ]:
def compute_cagr(nav_df, years):
    rows = []
    for amfi, group in nav_df.groupby('amfi_code'):
        group = group.sort_values('date')
        if group.empty:
            rows.append({'amfi_code': amfi, f'cagr_{years}yr': np.nan})
            continue
        end_date = group['date'].max()
        start_date = end_date - pd.DateOffset(years=years)
        candidate = group[group['date'] <= start_date] 
        if candidate.empty:
            rows.append({'amfi_code': amfi, f'cagr_{years}yr': np.nan})
            continue
        start_value = candidate.loc[candidate['date'].idxmax(), 'nav']
        end_value = group.loc[group['date'] == end_date, 'nav'].iloc[0]
        if start_value <= 0 or end_value <= 0:
            rows.append({'amfi_code': amfi, f'cagr_{years}yr': np.nan})
            continue
        cagr = (end_value / start_value) ** (1 / years) - 1
        rows.append({'amfi_code': amfi, f'cagr_{years}yr': cagr})
    return pd.DataFrame(rows)

cagr_1 = compute_cagr(nav_history, 1)
cagr_3 = compute_cagr(nav_history, 3)
cagr_5 = compute_cagr(nav_history, 5)
cagr_3.head()

## Performance scorecard and fund ranking

In [ ]:
fund_stats = scheme_performance[['amfi_code', 'scheme_name', 'expense_ratio_pct']].copy()
scorecard = fund_stats.merge(cagr_3, on='amfi_code', how='left').merge(metrics, on='amfi_code', how='left')
scorecard['drawdown'] = nav_history.groupby('amfi_code').apply(lambda g: (1 - g['nav'] / g['nav'].cummax()).max()).reset_index(level=0, drop=True).values
scorecard['cagr_rank'] = scorecard['cagr_3yr'].rank(pct=True, ascending=False)
scorecard['sharpe_rank'] = scorecard['sharpe_ratio'].rank(pct=True, ascending=False)
scorecard['expense_rank'] = 1 - scorecard['expense_ratio_pct'].rank(pct=True, ascending=True)
scorecard['drawdown_rank'] = 1 - scorecard['drawdown'].rank(pct=True, ascending=True)
scorecard['fund_score'] = (
    0.30 * scorecard['cagr_rank'] +
    0.25 * scorecard['sharpe_rank'] +
    0.20 * scorecard['sharpe_rank'] +
    0.15 * scorecard['expense_rank'] +
    0.10 * scorecard['drawdown_rank']
)
scorecard['rank'] = scorecard['fund_score'].rank(ascending=False)
scorecard.sort_values('rank').head()

## Save the scorecard as CSV

In [ ]:
scorecard_path = REPORTS_DIR / 'fund_scorecard.csv'
scorecard.to_csv(scorecard_path, index=False)
scorecard_path

## Benchmark comparison for the top 5 funds

In [ ]:
benchmarks = benchmark_indices.copy()
benchmarks['date'] = pd.to_datetime(benchmarks['date'], errors='coerce')
benchmarks = benchmarks.sort_values(['index_name', 'date'])
benchmarks['benchmark_return'] = benchmarks.groupby('index_name')['close_value'].pct_change()
benchmarks['benchmark_cumulative'] = benchmarks.groupby('index_name')['benchmark_return'].apply(lambda r: (1 + r.fillna(0)).cumprod() - 1)
top5 = scorecard.sort_values('fund_score', ascending=False).head(5)
comparison = []
for _, row in top5.iterrows():
    amfi = row['amfi_code']
    scheme = row['scheme_name']
    fund = nav_history[nav_history['amfi_code'] == amfi].copy()
    fund['fund_cumulative'] = (1 + fund['daily_return'].fillna(0)).cumprod() - 1",
50
100